In [2]:
import plotly.graph_objects as go
import networkx as nx
import sys
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import numpy as np

def load_graph(tissue_name, threshold=0.5):
    graph_path = f"tissue_networks/{tissue_name.replace(' ', '_')}_network.gexf"
    G = nx.read_gexf(graph_path)
    G_filtered = G.copy()
    edges_to_remove = [(u, v) for u, v, data in G_filtered.edges(data=True) 
                    if data.get('weight', 0) < threshold]
    G_filtered.remove_edges_from(edges_to_remove)
    return G_filtered

def load_embeddings(tissue_name):
    embeddings = pd.read_csv(f"tissue_embeddings/2d-umap/{tissue_name}_embeddings_2d.csv")
    names = embeddings['node'].tolist()
    emb_matrix = embeddings.drop(columns=['node']).to_numpy()
    print(emb_matrix.shape)
    return names, emb_matrix





In [3]:
def visualize_graph(tissue_name, threshold=0.5):
    G = load_graph(tissue_name, threshold)
    names, emb_matrix = load_embeddings(tissue_name)
    
    edge_x = []
    edge_y = []
    for u, v in G.edges():
        x0, y0 = emb_matrix[names.index(u)]
        x1, y1 = emb_matrix[names.index(v)]
        edge_x.extend([x0, x1, None])
        edge_y.extend([y0, y1, None])
    
    edge_trace = go.Scatter(
        x=edge_x, y=edge_y,
        line=dict(width=0.5, color='#888'),
        hoverinfo='none',
        mode='lines')
    
    node_x = []
    node_y = []
    for node in G.nodes():
        x, y = emb_matrix[names.index(node)]
        node_x.append(x)
        node_y.append(y)
    
    node_trace = go.Scatter(
        x=node_x, y=node_y,
        mode='markers',
        hoverinfo='text',
        marker=dict(
            showscale=True,
            colorscale='YlGnBu',
            reversescale=True,
            color=[],
            size=10,
            colorbar=dict(
                thickness=15,
                title='Node Connections',
                xanchor='left',
            ),
            line_width=2))
    
    node_adjacencies = []
    node_texts = []
    for node in G.nodes():
        adjacencies = len(list(G.adj[node]))
        node_adjacencies.append(adjacencies)
        node_texts.append(f'{node} has {adjacencies} connections')
    
    node_trace.marker.color = node_adjacencies
    node_trace.text = node_texts
    
    fig = go.Figure(data=[edge_trace, node_trace],
             layout=go.Layout(
                title=f'{tissue_name} Gene Co-expression Network (threshold={threshold})',
                showlegend=False,
                hovermode='closest',
                margin=dict(b=20,l=5,r=5,t=40),
                annotations=[ dict(
                    text="",
                    showarrow=False,
                    xref="paper", yref="paper") ],
                xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                yaxis=dict(showgrid=False, zeroline=False, showticklabels=False)
            ))
    print(f"{len(max(nx.connected_components(G), key=len))} nodes in largest connected component.")
    print(G.number_of_nodes())
    print(f"number of connected components: {nx.number_connected_components(G)}")
    fig.show()
visualize_graph("Liver", threshold=0.5)

(11095, 2)


KeyboardInterrupt: 

In [ ]:

def visualize_community_graph(tissue_name, threshold=0.5, resolution=1.0):
    G = load_graph(tissue_name=tissue_name, threshold=threshold)
    names, embeddings_2d = load_embeddings(tissue_name)
    print(f"loaded embeddings for {tissue_name}.")
    print(f"loading graph for {tissue_name}...")

    name_to_embedding = {name: emb for name, emb in zip(names, embeddings_2d)}

    communities = nx.community.louvain_communities(G, weight='weight', seed=42, resolution=resolution)
    print(f"Detected {len(communities)} communities in the graph.")
    community_graph = nx.Graph()
    pos = {}  

    for i, community in enumerate(communities):
        community_size = len(community)
        community_graph.add_node(i, size=community_size)
        
        community_embeddings = []
        for node in community:
            if node in name_to_embedding:
                community_embeddings.append(name_to_embedding[node])
        
        if community_embeddings:
            avg_embedding = np.mean(community_embeddings, axis=0)
            pos[i] = (avg_embedding[0], avg_embedding[1])
        else:
            pos[i] = (0, 0)
        
        for j in range(i + 1, len(communities)):
            weight = sum(1 for u in community for v in communities[j] if G.has_edge(u, v))
            if weight > 0:
                community_graph.add_edge(i, j, weight=weight)

    edge_x = []
    edge_y = []
    for edge in community_graph.edges():
        x0, y0 = pos[edge[0]]
        x1, y1 = pos[edge[1]]
        edge_x.append(x0)
        edge_x.append(x1)
        edge_x.append(None)
        edge_y.append(y0)
        edge_y.append(y1)
        edge_y.append(None)

    edge_trace = go.Scatter(
        x=edge_x, y=edge_y,
        line=dict(width=0.5, color='#888'),
        hoverinfo='none',
        mode='lines')
    # Nodes
    node_x = []
    node_y = []
    node_sizes = []
    for node in community_graph.nodes():
        x, y = pos[node]
        node_x.append(x)
        node_y.append(y)
        node_sizes.append(community_graph.nodes[node]['size'])

    node_trace = go.Scatter(
        x=node_x, y=node_y,
        mode='markers',
        hoverinfo='text',
        text=[f"Community {i}: {size} nodes" for i, size in enumerate(node_sizes)],
        marker=dict(
            showscale=True,
            colorscale='YlGnBu',
            size=node_sizes,  # Size based on community size
            sizemode='area',
            sizeref=2.*max(node_sizes)/(40.**2),  # Scale for visibility
            sizemin=4,
            line_width=2))

    fig = go.Figure(data=[edge_trace, node_trace],
        layout=go.Layout(
            title=f"{tissue_name} Network Community Graph",
            showlegend=False,
            hovermode='closest',
            margin=dict(b=20,l=5,r=5,t=40),
            xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
            yaxis=dict(showgrid=False, zeroline=False, showticklabels=False)
        ))
    
    print(f"{len(max(nx.connected_components(G), key=len))} nodes in largest connected component.")
    print(G.number_of_nodes())
    print(f"number of connected components: {nx.number_connected_components(G)}")

    fig.show()
tissue_name = "Liver"
threshold = 0.0
visualize_community_graph(tissue_name, threshold)

(11095, 2)
loaded embeddings for Liver.
loading graph for Liver...
Detected 121 communities in the graph.
10934 nodes in largest connected component.
10934
number of connected components: 1


In [ ]:
def evaluate_coursened_graph_sizes(tissue_name, threshold=0.5):
    G = load_graph(tissue_name=tissue_name, threshold=threshold)
    print(f"loading graph for {tissue_name}...")
    metrics = {
        "original": (G.number_of_nodes(), G.number_of_edges()),
        'coarsened': []
    }
    for coarsening_factor in [1, 2, 5, 10, 20]:
        print(f"Evaluating coarsened graph with factor {coarsening_factor}...")
        communities = nx.community.louvain_communities(G, weight='weight', seed=42, resolution=coarsening_factor)


        community_graph = nx.Graph()
        pos = {}  

        for i, community in enumerate(communities):
            community_size = len(community)
            community_graph.add_node(i, size=community_size)
            for j in range(i + 1, len(communities)):
                weight = sum(1 for u in community for v in communities[j] if G.has_edge(u, v))
                if weight > 0:
                    community_graph.add_edge(i, j, weight=weight)
        num_nodes = community_graph.number_of_nodes()
        num_edges = community_graph.number_of_edges()
        print(f"Coarsened graph with factor {coarsening_factor}: {num_nodes} nodes, {num_edges} edges")
        metrics['coarsened'].append((coarsening_factor, num_nodes, num_edges))
    return metrics


evaluate_coursened_graph_sizes("Liver", threshold=0.0)


loading graph for Liver...
Evaluating coarsened graph with factor 1...
Coarsened graph with factor 1: 58 nodes, 72 edges
Evaluating coarsened graph with factor 2...
Coarsened graph with factor 2: 82 nodes, 111 edges
Evaluating coarsened graph with factor 5...
Coarsened graph with factor 5: 121 nodes, 182 edges
Evaluating coarsened graph with factor 10...
Coarsened graph with factor 10: 176 nodes, 278 edges
Evaluating coarsened graph with factor 20...
Coarsened graph with factor 20: 265 nodes, 456 edges


In [10]:
def evaluate_coarsened_graph_sizes(tissue_name, threshold=0.5):
    G = load_graph(tissue_name=tissue_name, threshold=threshold)
    print(f"Loading graph for {tissue_name}...")
    metrics = {
        "original": (G.number_of_nodes(), G.number_of_edges()),
        'coarsened': []
    }
    for coarsening_factor in [1, 2, 5, 10, 20]:
        print(f"Evaluating coarsened graph with factor {coarsening_factor}...")
        communities = nx.community.louvain_communities(G, weight='weight', seed=42, resolution=coarsening_factor)
        community_graph = nx.Graph()
        
        for i, community in enumerate(communities):
            community_size = len(community)
            community_graph.add_node(i, size=community_size)
            for j in range(i + 1, len(communities)):
                weight = sum(1 for u in community for v in communities[j] if G.has_edge(u, v))
                if weight > 0:
                    community_graph.add_edge(i, j, weight=weight)
        
        num_nodes = community_graph.number_of_nodes()
        num_edges = community_graph.number_of_edges()
        print(f"Coarsened graph with factor {coarsening_factor}: {num_nodes} nodes, {num_edges} edges")
        metrics['coarsened'].append((coarsening_factor, num_nodes, num_edges))
    
    return metrics

# Collect all results
results = []
for file in os.listdir('tissue_networks'):
    if file.endswith("_network.gexf"):
        # Extract tissue name from filename
        tissue_name = file.replace("_network.gexf", "")
        metrics = evaluate_coarsened_graph_sizes(tissue_name, threshold=0.0)
        
        # Build row for DataFrame
        row = {"Tissue": tissue_name}
        for factor, num_nodes, num_edges in metrics['coarsened']:
            row[f"num_nodes_{factor}"] = num_nodes
            row[f"num_edges_{factor}"] = num_edges
        results.append(row)

# Create DataFrame and save
df = pd.DataFrame(results)
df.to_csv("coarsened_graph_metrics.csv", index=False)

NameError: name 'os' is not defined

In [ ]:
x = 10699 - 12376

print(x)


In [ ]:

print(len(max(nx.connected_components(G), key=len)))
print(G.number_of_nodes())

things i could do as a follow-up
- drop nodes that arent connected
- plot with FR and community coloring
- run K means on the embedded data and compare to k-means
- try different community detection algorithms
- keep analytics for each tissue